<a href="https://colab.research.google.com/github/90splayer/Applied-ML/blob/main/Unbiased_Multiple_Instance_Learning_for_Video_Anomaly_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🔹 STEP 1: Prepare your data (MOST IMPORTANT)

In [ ]:
data = [
    (features, label),
    (features, label),
]

In [ ]:
import os
import numpy as np

def load_dataset(path):
    data = []

    for label_name in ["normal", "anomaly"]:
        label = 0 if label_name == "normal" else 1
        folder = os.path.join(path, label_name)

        for file in os.listdir(folder):
            if file.endswith(".npy"):
                features = np.load(os.path.join(folder, file))
                data.append((features, label))

    return data

data = load_dataset("dataset/")

🔹 STEP 2: Convert videos → segments (MIL format)

In [ ]:
T = 32

def split_features(features, T):
    segment_size = len(features) // T
    segments = []

    for i in range(T):
        segment = features[i*segment_size:(i+1)*segment_size]
        segments.append(segment.mean(axis=0))

    return np.array(segments)

In [ ]:
processed_data = []

for features, label in data:
    segments = split_features(features, T)
    processed_data.append((segments, label))

🔹 STEP 3: Train / Test split

In [ ]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(processed_data, test_size=0.2)

🔹 STEP 4: Create batches (VERY IMPORTANT for MIL)

In [ ]:
import torch

def get_batches(data, batch_size=8):
    np.random.shuffle(data)

    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]

        normal = [x[0] for x in batch if x[1] == 0]
        abnormal = [x[0] for x in batch if x[1] == 1]

        if len(normal) == 0 or len(abnormal) == 0:
            continue

        yield torch.tensor(normal, dtype=torch.float32), \
              torch.tensor(abnormal, dtype=torch.float32)

🔹 STEP 5: Build model

In [ ]:
import torch.nn as nn

class AttentionMIL(nn.Module):
    def __init__(self, input_dim=1024):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.classifier = nn.Linear(input_dim, 1)

    def forward(self, x):
        attn = torch.softmax(self.attention(x), dim=1)
        scores = self.classifier(x).squeeze(-1)
        return scores, attn

🔹 STEP 6: 🔥 Unbiased MIL Loss

In [ ]:
def topk_mean(scores, k):
    topk, _ = torch.topk(scores, k, dim=1)
    return torch.mean(topk, dim=1)

In [ ]:
def unbiased_mil_loss(normal_scores, abnormal_scores, k=5):

    topk_abnormal = topk_mean(abnormal_scores, k)
    topk_normal = topk_mean(normal_scores, k)

    ranking_loss = torch.mean(torch.clamp(1 - topk_abnormal + topk_normal, min=0))

    sparsity_loss = torch.mean(abnormal_scores)

    return ranking_loss + 0.0001 * sparsity_loss

🔹 STEP 7: Training loop (PUT IT ALL TOGETHER)

In [ ]:
model = AttentionMIL()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for normal_batch, abnormal_batch in get_batches(train_data):

        normal_scores, _ = model(normal_batch)
        abnormal_scores, _ = model(abnormal_batch)

        loss = unbiased_mil_loss(normal_scores, abnormal_scores)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

🔹 STEP 8: Evaluation (AUC)

In [ ]:
from sklearn.metrics import roc_auc_score

model.eval()
preds = []
labels = []

with torch.no_grad():
    for segments, label in test_data:
        segments = torch.tensor(segments).unsqueeze(0).float()

        scores, _ = model(segments)

        video_score = torch.max(scores)  # MIL rule

        preds.append(video_score.item())
        labels.append(label)

auc = roc_auc_score(labels, preds)
print("AUC:", auc)

🔹 STEP 9: Visualisation (REPORT MARKS)

In [ ]:
import matplotlib.pyplot as plt

scores = scores.squeeze().numpy()

plt.plot(scores)
plt.title("Anomaly Scores Over Time")
plt.xlabel("Segments")
plt.ylabel("Score")
plt.show()